In [0]:
# ===================================================
# BLOCK 1 — IMPORTS AND FACT CONFIGURATION (PYTHON)
# ===================================================

"""
Define the source-backed yield fact targets. Retest and FTY columns remain null
until the explicit simulated retest source is added on Day 3.
"""

from pyspark.sql import functions as F

CATALOG = "semiconplus_portfolio"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"

SILVER_PRODUCTION_LOTS = f"{CATALOG}.{SILVER_SCHEMA}.production_lots"
DIM_DATE = f"{CATALOG}.{GOLD_SCHEMA}.dim_date"
DIM_SITE = f"{CATALOG}.{GOLD_SCHEMA}.dim_site"
DIM_PRODUCT_GROUP = f"{CATALOG}.{GOLD_SCHEMA}.dim_product_group"
DIM_DEVICE = f"{CATALOG}.{GOLD_SCHEMA}.dim_device"
DIM_EQUIPMENT = f"{CATALOG}.{GOLD_SCHEMA}.dim_equipment"
DIM_LOT = f"{CATALOG}.{GOLD_SCHEMA}.dim_lot"

FACT_LOT_PERFORMANCE = f"{CATALOG}.{GOLD_SCHEMA}.fact_lot_performance"
FACT_YIELD_PERIODIC = f"{CATALOG}.{GOLD_SCHEMA}.fact_yield_periodic"

spark.conf.set("spark.sql.session.timeZone", "UTC")

print("Day 2 source-backed fact configuration loaded.")


In [0]:
# ===================================================
# BLOCK 2 — BUILD THE SOURCE-BACKED LOT FACT (PYTHON)
# ===================================================

"""
Publish one row per source lot using authoritative production quantities.
Retest-derived metrics are deliberately null until Day 3.
"""

lot_fact_df = (
    spark.table(SILVER_PRODUCTION_LOTS).alias("source")
    .join(
        spark.table(DIM_LOT).alias("lot"),
        F.col("source.lot_id") == F.col("lot.source_lot_id"),
        "inner",
    )
    .join(
        spark.table(DIM_DATE).alias("date"),
        F.col("lot.production_date") == F.col("date.full_date"),
        "left",
    )
    .select(
        F.col("lot.lot_key"),
        F.col("date.date_key"),
        F.col("lot.site_key"),
        F.col("lot.product_group_key"),
        F.col("lot.device_key"),
        F.col("lot.equipment_key"),
        F.col("source.lot_id").alias("source_lot_id"),
        F.col("lot.test_batch_id"),
        F.col("lot.test_lot_id"),
        F.col("lot.production_date"),
        F.col("source.quantity_started").cast("long").alias("input_quantity"),
        F.col("source.quantity_passed")
        .cast("long")
        .alias("first_pass_good_quantity"),
        F.col("source.quantity_failed")
        .cast("long")
        .alias("first_pass_fail_quantity"),
        F.when(
            F.col("source.quantity_started") > 0,
            F.col("source.quantity_passed").cast("double")
            / F.col("source.quantity_started").cast("double"),
        ).alias("first_pass_yield"),
        F.lit(None).cast("long").alias("retest_input_quantity"),
        F.lit(None).cast("long").alias("retest_good_quantity"),
        F.lit(None).cast("long").alias("retest_fail_quantity"),
        F.lit(None).cast("long").alias("final_good_quantity"),
        F.lit(None).cast("double").alias("final_test_yield"),
        F.lit(None).cast("double").alias("retest_recovery_contribution"),
        F.lit(False).alias("retest_data_available_flag"),
        (F.col("source.quantity_started") > 0).alias("tested_lot_flag"),
        F.when(
            F.col("source.quantity_started") <= 0,
            F.lit("NOT_TESTED"),
        )
        .when(
            (
                F.col("source.quantity_passed").cast("double")
                / F.col("source.quantity_started").cast("double")
            ) >= F.lit(0.95),
            F.lit("FPY_TARGET_MET"),
        )
        .otherwise(F.lit("FPY_BELOW_TARGET"))
        .alias("first_pass_status"),
        F.current_timestamp().alias("_gold_processed_at_utc"),
    )
)

(
    lot_fact_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(FACT_LOT_PERFORMANCE)
)

print(f"Lot-performance fact published: {lot_fact_df.count():,}")

In [0]:
# ===================================================
# BLOCK 3 — BUILD THE PERIODIC YIELD FACT (PYTHON)
# ===================================================

"""
Aggregate authoritative lot quantities at the reporting grain. Ratios are
calculated from summed quantities, never from averaged lot percentages.
"""

periodic_fact_df = (
    spark.table(FACT_LOT_PERFORMANCE)
    .groupBy(
        "date_key",
        "production_date",
        "site_key",
        "product_group_key",
        "device_key",
    )
    .agg(
        F.count("*").alias("lot_count"),
        F.sum(F.col("tested_lot_flag").cast("long")).alias("tested_lot_count"),
        F.sum("input_quantity").alias("input_quantity"),
        F.sum("first_pass_good_quantity").alias("first_pass_good_quantity"),
        F.sum("first_pass_fail_quantity").alias("first_pass_fail_quantity"),
        F.sum(
            F.when(
                F.col("first_pass_status") == "FPY_BELOW_TARGET",
                F.lit(1),
            ).otherwise(F.lit(0))
        ).alias("first_pass_below_target_lot_count"),
    )
    .withColumn(
        "first_pass_yield",
        F.when(
            F.col("input_quantity") > 0,
            F.col("first_pass_good_quantity").cast("double")
            / F.col("input_quantity").cast("double"),
        ),
    )
    .withColumn("retest_input_quantity", F.lit(None).cast("long"))
    .withColumn("retest_good_quantity", F.lit(None).cast("long"))
    .withColumn("retest_fail_quantity", F.lit(None).cast("long"))
    .withColumn("final_good_quantity", F.lit(None).cast("long"))
    .withColumn("final_test_yield", F.lit(None).cast("double"))
    .withColumn(
        "retest_recovery_contribution",
        F.lit(None).cast("double"),
    )
    .withColumn("retest_data_available_flag", F.lit(False))
    .withColumn("_gold_processed_at_utc", F.current_timestamp())
)

(
    periodic_fact_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(FACT_YIELD_PERIODIC)
)

print(f"Periodic-yield fact published: {periodic_fact_df.count():,}")
display(periodic_fact_df.orderBy("production_date").limit(20))

In [0]:
# ===================================================
# BLOCK 4 — FACT BUILD RESULT (PYTHON)
# ===================================================

fact_results = [
    (FACT_LOT_PERFORMANCE, spark.table(FACT_LOT_PERFORMANCE).count()),
    (FACT_YIELD_PERIODIC, spark.table(FACT_YIELD_PERIODIC).count()),
]

display(
    spark.createDataFrame(
        fact_results,
        ["table_name", "row_count"],
    )
)

print("DAY 2 SOURCE-BACKED YIELD FACT BUILD: COMPLETED")